# SUHU BIM avatar pilot

Run this notebook from top to bottom in a Google Colab GPU runtime. It validates `Suhu.mp4`, runs pinned SMPLer-X inference one frame at a time, persists every completed artifact in Google Drive, reconstructs a neutral SMPL-X avatar, and applies the three-reviewer BIM acceptance gate.

This is pretrained inference, not training. Use one Google account and do not use the company eKYC server. A Colab disconnect is safe after any frame copied to Drive.

## Before running

Confirm hackathon non-commercial demonstration rights, permission to use the BIM source video, and the current terms for SMPLer-X, SMPL-X, and each checkpoint. Restricted videos and models stay in private Drive and must never be committed or redistributed. Obtain SMPL-X files from the official site after registration; obtain SMPLer-X and MMDetection checkpoints from their official project links.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import csv
import hashlib
import json
import math
import os
import shutil
import subprocess
import sys
import time

DRIVE_ROOT = Path('/content/drive/MyDrive/BIM-Avatar')
RUN_DIR = DRIVE_ROOT / 'runs/SUHU'
INPUT_VIDEO = DRIVE_ROOT / 'inputs/Suhu.mp4'
PILOT_REPO = Path('/content/SignAvatar-Muba')
PILOT_REPO_URL = 'https://github.com/Chee613/SignAvatar-Muba.git'
PILOT_REPO_REF = 'main'
SMPLERX_DIR = Path('/content/SMPLer-X')
SMPLERX_COMMIT = '064baef0e4ab5277a3297691bc1d46ea5412586f'
MODEL_NAME = 'smpler_x_h32'
REPRESENTATIVE_FRAME = 38
COMMAND_LOG = []

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()

def run(command, cwd=None, check=True, capture_output=False):
    command = [str(value) for value in command]
    started = utc_now()
    result = subprocess.run(command, cwd=cwd, check=check, text=True, capture_output=capture_output)
    COMMAND_LOG.append({'command': command, 'cwd': str(cwd) if cwd else None, 'started_at': started, 'returncode': result.returncode})
    return result

def atomic_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, indent=2) + '\n', encoding='utf-8')
    os.replace(temporary, path)

def copy_once(source, destination):
    source, destination = Path(source), Path(destination)
    if destination.exists():
        return False
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(destination.suffix + '.tmp')
    shutil.copy2(source, temporary)
    os.replace(temporary, destination)
    return True

def update_manifest(**values):
    path = RUN_DIR / 'manifest.json'
    manifest = json.loads(path.read_text(encoding='utf-8')) if path.exists() else {}
    manifest.update(values)
    manifest.setdefault('created_at', utc_now())
    manifest['updated_at'] = utc_now()
    manifest['commands'] = manifest.get('commands', []) + COMMAND_LOG
    COMMAND_LOG.clear()
    atomic_json(path, manifest)
    return manifest

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

for relative in ('frames', 'smplx', 'meta', 'overlays', 'motion', 'qa', 'avatar', 'avatar/frames'):
    (RUN_DIR / relative).mkdir(parents=True, exist_ok=True)
update_manifest(sign_id='SUHU', drive_root=str(DRIVE_ROOT))
print('Persistent run folder:', RUN_DIR)

In [ ]:
gpu_line = subprocess.check_output([
    'nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader,nounits'
]).decode().strip().splitlines()[0]
gpu_name, gpu_total, gpu_free = [value.strip() for value in gpu_line.split(',')]
meminfo = {}
for line in Path('/proc/meminfo').read_text().splitlines():
    key, value = line.split(':', 1)
    meminfo[key] = int(value.strip().split()[0])
available_ram_gib = meminfo['MemAvailable'] / 1024 / 1024
free_disk_gib = shutil.disk_usage('/content').free / 1024 ** 3

checks = {
    'tesla_t4': 'T4' in gpu_name,
    'free_vram_at_least_14_gib': int(gpu_free) >= 14 * 1024,
    'available_ram_at_least_10_gib': available_ram_gib >= 10,
    'temporary_disk_at_least_20_gib': free_disk_gib >= 20,
}
preflight = {
    'gpu_name': gpu_name, 'gpu_total_mib': int(gpu_total), 'gpu_free_mib': int(gpu_free),
    'available_ram_gib': round(available_ram_gib, 2), 'free_disk_gib': round(free_disk_gib, 2),
    'checks': checks,
}
update_manifest(colab_preflight=preflight)
print(json.dumps(preflight, indent=2))
if not all(checks.values()):
    raise RuntimeError('Colab preflight failed. Reconnect to a fresh T4 runtime before installing models.')

In [ ]:
if not PILOT_REPO.exists():
    run(['git', 'clone', PILOT_REPO_URL, PILOT_REPO])
run(['git', 'fetch', '--all', '--tags'], cwd=PILOT_REPO)
run(['git', 'checkout', PILOT_REPO_REF], cwd=PILOT_REPO)

if not SMPLERX_DIR.exists():
    run(['git', 'clone', 'https://github.com/caizhongang/SMPLer-X.git', SMPLERX_DIR])
run(['git', 'checkout', SMPLERX_COMMIT], cwd=SMPLERX_DIR)
inference_script = SMPLERX_DIR / 'main/inference.py'
inference_source = inference_script.read_text(encoding='utf-8')
main_anchor = 'def main():\n'
cwd_guard = '    os.chdir(osp.dirname(osp.abspath(__file__)))\n'
if cwd_guard not in inference_source:
    if inference_source.count(main_anchor) != 1:
        raise RuntimeError('Cannot apply the pinned SMPLer-X working-directory compatibility patch.')
    inference_script.write_text(inference_source.replace(main_anchor, main_anchor + cwd_guard, 1), encoding='utf-8')

micromamba = Path('/content/bin/micromamba')
if not micromamba.exists():
    archive = Path('/content/micromamba.tar.bz2')
    run(['curl', '-Ls', 'https://micro.mamba.pm/api/micromamba/linux-64/latest', '-o', archive])
    run(['tar', '-xjf', archive, '-C', '/content', 'bin/micromamba'])
env_prefix = Path('/content/envs/smplerx')
env_python = env_prefix / 'bin/python'
if not env_python.exists():
    run([micromamba, 'create', '-y', '-p', env_prefix, '-c', 'pytorch', '-c', 'conda-forge',
         'python=3.8', 'pip', 'pytorch=1.12.0', 'torchvision=0.13.0', 'torchaudio=0.12.0',
         'cudatoolkit=11.3', 'mkl=2024.0'])

install_marker = env_prefix / '.suhu_dependencies_installed'
install_signature = SMPLERX_COMMIT + '\nmkl=2024.0\nyapf=0.40.1\ntorchgeometry-bool-mask=v1\n'
if not install_marker.exists() or install_marker.read_text() != install_signature:
    run([micromamba, 'install', '-y', '-p', env_prefix, '-c', 'pytorch', '-c', 'conda-forge', 'mkl=2024.0'])
    run([env_python, '-m', 'pip', 'install', 'numpy==1.23.5'])
    run([env_python, '-m', 'pip', 'install', 'mmcv-full==1.7.1',
         '-f', 'https://download.openmmlab.com/mmcv/dist/cu113/torch1.12.0/index.html'])
    run([env_python, '-m', 'pip', 'install', '-r', SMPLERX_DIR / 'requirements.txt'])
    run([env_python, '-m', 'pip', 'install', '-v', '-e', SMPLERX_DIR / 'main/transformer_utils'])
    run([env_python, '-m', 'pip', 'install', 'yapf==0.40.1'])
    torchgeometry_path = env_prefix / 'lib/python3.8/site-packages/torchgeometry/core/conversions.py'
    torchgeometry_source = torchgeometry_path.read_text(encoding='utf-8')
    torchgeometry_original = ('    mask_c1 = mask_d2 * (1 - mask_d0_d1)\n'
                              '    mask_c2 = (1 - mask_d2) * mask_d0_nd1\n'
                              '    mask_c3 = (1 - mask_d2) * (1 - mask_d0_nd1)\n')
    torchgeometry_fixed = ('    mask_c1 = mask_d2 * (~mask_d0_d1)\n'
                           '    mask_c2 = (~mask_d2) * mask_d0_nd1\n'
                           '    mask_c3 = (~mask_d2) * (~mask_d0_nd1)\n')
    if torchgeometry_original in torchgeometry_source:
        torchgeometry_path.write_text(torchgeometry_source.replace(torchgeometry_original, torchgeometry_fixed, 1), encoding='utf-8')
    elif torchgeometry_fixed not in torchgeometry_source:
        raise RuntimeError('Cannot apply the TorchGeometry boolean-mask compatibility patch.')
    install_marker.write_text(install_signature)

verification = run([env_python, '-c',
    "import json, cv2, mmcv, mmdet, smplx, torch, yapf; print(json.dumps({'torch': torch.__version__, 'cuda': torch.version.cuda, 'cuda_available': torch.cuda.is_available(), 'mmdet': mmdet.__version__, 'mmcv': mmcv.__version__, 'opencv': cv2.__version__, 'smplx': getattr(smplx, '__version__', '0.1.28'), 'yapf': yapf.__version__}))"
], capture_output=True)
versions = json.loads(verification.stdout.strip().splitlines()[-1])
if not versions['cuda_available']:
    raise RuntimeError('The isolated SMPLer-X environment cannot access CUDA.')
sys.path.insert(0, str(PILOT_REPO))
from suhu_pilot import analyze_motion, assess_reviews, consolidate_motion, first_unprocessed_frame, load_config, reconcile_progress, validate_file, validate_source_probe
config = load_config(PILOT_REPO / 'config/suhu_pilot.json')
update_manifest(upstream_commit=SMPLERX_COMMIT, checkpoint_name=MODEL_NAME, dependency_versions=versions, pilot_config=config)
print(json.dumps(versions, indent=2))

In [ ]:
public_assets = [
    {
        'path': DRIVE_ROOT / 'models/smplerx/smpler_x_h32.pth.tar',
        'url': 'https://huggingface.co/caizhongang/SMPLer-X/resolve/e121949b1a4a3adc0b447eab97a2a21694dffbef/smpler_x_h32.pth.tar?download=true',
        'size': 7942394512,
        'sha256': 'efb86805527e68cd9e37605d66d9f02e25c56a0bf933568360ca02f85b3f5dcf',
    },
    {
        'path': DRIVE_ROOT / 'models/mmdet/faster_rcnn_r50_fpn_1x_coco_20200130-047c8118.pth',
        'url': 'https://download.openmmlab.com/mmdetection/v2.0/faster_rcnn/faster_rcnn_r50_fpn_1x_coco/faster_rcnn_r50_fpn_1x_coco_20200130-047c8118.pth',
        'size': 167287506,
        'sha256': '047c8118fc5ca88ba5ae1fab72f2cd6b070501fe3af2f3cba5cfa9a89b44b03e',
    },
    {
        'path': DRIVE_ROOT / 'models/mmdet/mmdet_faster_rcnn_r50_fpn_coco.py',
        'url': 'https://raw.githubusercontent.com/openxrlab/xrmocap/2d227a7b27cfec5c43bfe975d23bc0a54d24541a/configs/modules/human_perception/mmdet_faster_rcnn_r50_fpn_coco.py',
        'size': 5867,
        'sha256': 'bb2905aaf5ff6aa4ce75d9aa63070368a96e9add16e3baab28debdea7c064c48',
    },
]
verified_public_assets = []
for asset in public_assets:
    path = asset['path']
    path.parent.mkdir(parents=True, exist_ok=True)
    valid = False
    if path.exists() and path.stat().st_size == asset['size']:
        try:
            verified_public_assets.append(validate_file(path, asset['size'], asset['sha256']))
            valid = True
        except ValueError:
            invalid = path.with_name(path.name + '.invalid-' + datetime.now().strftime('%Y%m%d-%H%M%S'))
            path.rename(invalid)
            print(f'Preserved invalid download as {invalid.name}.')
    elif path.exists() and path.stat().st_size > asset['size']:
        invalid = path.with_name(path.name + '.invalid-' + datetime.now().strftime('%Y%m%d-%H%M%S'))
        path.rename(invalid)
        print(f'Preserved oversized download as {invalid.name}.')
    if not valid:
        print(f"Downloading {path.name}; existing partial files resume automatically.")
        run(['curl', '--location', '--fail', '--retry', '5', '--retry-all-errors',
             '--continue-at', '-', '--output', path, asset['url']])
        verified_public_assets.append(validate_file(path, asset['size'], asset['sha256']))
update_manifest(public_assets=verified_public_assets)
print('Public SMPLer-X and MMDetection assets are complete and verified.')

In [ ]:
private_models = DRIVE_ROOT / 'models'
required = {
    private_models / f'smplerx/{MODEL_NAME}.pth.tar': SMPLERX_DIR / f'pretrained_models/{MODEL_NAME}.pth.tar',
    private_models / 'mmdet/faster_rcnn_r50_fpn_1x_coco_20200130-047c8118.pth': SMPLERX_DIR / 'pretrained_models/mmdet/faster_rcnn_r50_fpn_1x_coco_20200130-047c8118.pth',
    private_models / 'mmdet/mmdet_faster_rcnn_r50_fpn_coco.py': SMPLERX_DIR / 'pretrained_models/mmdet/mmdet_faster_rcnn_r50_fpn_coco.py',
}
for name in ('MANO_SMPLX_vertex_ids.pkl', 'SMPL-X__FLAME_vertex_ids.npy', 'SMPLX_NEUTRAL.pkl',
             'SMPLX_to_J14.pkl', 'SMPLX_NEUTRAL.npz', 'SMPLX_MALE.npz', 'SMPLX_FEMALE.npz'):
    required[private_models / 'smplx' / name] = SMPLERX_DIR / 'common/utils/human_model_files/smplx' / name
for name in ('SMPL_NEUTRAL.pkl', 'SMPL_MALE.pkl', 'SMPL_FEMALE.pkl'):
    required[private_models / 'smplx/smpl' / name] = SMPLERX_DIR / 'common/utils/human_model_files/smpl' / name
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError('Required private files are missing from Drive:\n' + '\n'.join(missing))
for source, destination in required.items():
    destination.parent.mkdir(parents=True, exist_ok=True)
    if not destination.exists() or destination.stat().st_size != source.stat().st_size:
        shutil.copy2(source, destination)
model_hashes = {str(source.relative_to(DRIVE_ROOT)): sha256_file(source) for source in required}
update_manifest(model_file_hashes=model_hashes)
print(f'Validated and staged {len(required)} private model files.')

In [ ]:
if not INPUT_VIDEO.is_file():
    raise FileNotFoundError(f'Put the private source video at {INPUT_VIDEO}')
source_hash = sha256_file(INPUT_VIDEO)
probe_result = run(['ffprobe', '-v', 'error', '-show_streams', '-show_format', '-of', 'json', INPUT_VIDEO], capture_output=True)
probe = json.loads(probe_result.stdout)
validated_source = validate_source_probe(probe, config)
update_manifest(source={'path': str(INPUT_VIDEO), 'sha256': source_hash, **validated_source}, ffmpeg_version=subprocess.check_output(['ffmpeg', '-version']).decode().splitlines()[0])
print(json.dumps(validated_source, indent=2))

In [ ]:
source_30fps = RUN_DIR / 'suhu_original_30fps.mp4'
if not source_30fps.exists():
    run(['ffmpeg', '-y', '-i', INPUT_VIDEO, '-an', '-vf', f"fps={config['processing_fps']}",
         '-c:v', 'libx264', '-pix_fmt', 'yuv420p', source_30fps])
frames_dir = RUN_DIR / 'frames'
frames = sorted(frames_dir.glob('*.jpg'))
expected_derived = round(config['expected_source_frames'] * config['processing_fps'] / config['source_fps'])
if not frames:
    run(['ffmpeg', '-y', '-i', source_30fps, '-q:v', '2', frames_dir / '%06d.jpg'])
    frames = sorted(frames_dir.glob('*.jpg'))
actual_names = [path.name for path in frames]
expected_names = [f'{frame:06d}.jpg' for frame in range(1, len(frames) + 1)]
if actual_names != expected_names or abs(len(frames) - expected_derived) > 1:
    raise RuntimeError(f'Invalid derived frames: expected about {expected_derived} contiguous files, found {len(frames)}')
if sha256_file(INPUT_VIDEO) != source_hash:
    raise RuntimeError('The original source video changed during preprocessing.')
total_frames = len(frames)
update_manifest(processing_fps=config['processing_fps'], derived_frame_count=total_frames, generated_artifacts={'source_30fps': str(source_30fps), 'frames': str(frames_dir)})
print(f'Prepared {total_frames} contiguous frames; original SHA-256 is unchanged.')

In [ ]:
representative_dir = RUN_DIR / 'qa/representative_frame'
representative_outputs = [
    representative_dir / f'smplx/{REPRESENTATIVE_FRAME:05d}_0.npz',
    representative_dir / f'meta/{REPRESENTATIVE_FRAME:05d}_0.json',
    representative_dir / f'img/{REPRESENTATIVE_FRAME:06d}.jpg',
]
if not all(path.exists() for path in representative_outputs):
    before = subprocess.check_output(['nvidia-smi', '--query-gpu=memory.free', '--format=csv,noheader,nounits']).decode().strip()
    command = [env_python, 'inference.py', '--num_gpus', '1', '--pretrained_model', MODEL_NAME,
               '--img_path', RUN_DIR / 'frames', '--start', REPRESENTATIVE_FRAME, '--end', '1',
               '--output_folder', representative_dir]
    result = run(command, cwd=SMPLERX_DIR / 'main', check=False, capture_output=True)
    if result.returncode != 0:
        failure_log = RUN_DIR / 'qa/representative_frame.log'
        failure_log.write_text(result.stdout + result.stderr, encoding='utf-8')
        update_manifest(representative_frame_failure={'frame': REPRESENTATIVE_FRAME, 'returncode': result.returncode, 'log': str(failure_log)})
        raise RuntimeError(f'Representative-frame inference failed. Inspect {failure_log}.\n{result.stderr[-4000:]}')
    after = subprocess.check_output(['nvidia-smi', '--query-gpu=memory.free', '--format=csv,noheader,nounits']).decode().strip()
    update_manifest(representative_frame={'frame': REPRESENTATIVE_FRAME, 'vram_free_mib_before': before, 'vram_free_mib_after': after})
if not all(path.exists() for path in representative_outputs):
    raise RuntimeError('Representative-frame inference did not create SMPL-X, metadata, and overlay outputs.')
from IPython.display import display
from PIL import Image
display(Image.open(representative_outputs[2]))
print('Inspect body alignment, wrist orientation, finger configuration, and hand contact before continuing.')

In [ ]:
progress = reconcile_progress(RUN_DIR, total_frames)
start_frame = first_unprocessed_frame(progress, total_frames)
if start_frame is None:
    print('All frames already have terminal statuses.')
else:
    local_output = Path('/content/suhu_inference')
    if local_output.exists():
        shutil.rmtree(local_output)
    local_output.mkdir()
    log_path = RUN_DIR / 'qa/inference.log'
    command = [str(env_python), 'inference.py', '--num_gpus', '1', '--pretrained_model', MODEL_NAME,
               '--img_path', str(RUN_DIR / 'frames'), '--start', str(start_frame),
               '--end', str(total_frames - start_frame + 1), '--output_folder', str(local_output)]
    started = utc_now()
    with log_path.open('a', encoding='utf-8') as log:
        process = subprocess.Popen(command, cwd=SMPLERX_DIR / 'main', stdout=log, stderr=subprocess.STDOUT, text=True)
        while process.poll() is None:
            for frame in range(start_frame, total_frames + 1):
                if (local_output / 'img' / f'{frame:06d}.jpg').exists():
                    for folder, name in (('smplx', f'{frame:05d}_0.npz'), ('meta', f'{frame:05d}_0.json'), ('img', f'{frame:06d}.jpg')):
                        source = local_output / folder / name
                        destination_folder = 'overlays' if folder == 'img' else folder
                        if source.exists():
                            copy_once(source, RUN_DIR / destination_folder / name)
            reconcile_progress(RUN_DIR, total_frames)
            time.sleep(1)
    for frame in range(start_frame, total_frames + 1):
        if (local_output / 'img' / f'{frame:06d}.jpg').exists():
            for folder, name in (('smplx', f'{frame:05d}_0.npz'), ('meta', f'{frame:05d}_0.json'), ('img', f'{frame:06d}.jpg')):
                source = local_output / folder / name
                destination_folder = 'overlays' if folder == 'img' else folder
                if source.exists():
                    copy_once(source, RUN_DIR / destination_folder / name)
    COMMAND_LOG.append({'command': command, 'cwd': str(SMPLERX_DIR / 'main'), 'started_at': started, 'returncode': process.returncode})
    progress = reconcile_progress(RUN_DIR, total_frames)
    if process.returncode != 0:
        update_manifest(inference_failure={'returncode': process.returncode, 'log': str(log_path), 'resume_from': first_unprocessed_frame(progress, total_frames)})
        raise RuntimeError(f'Inference stopped with code {process.returncode}. Inspect {log_path}; completed frames remain safe in Drive.')
statuses = [progress['frames'][str(frame)]['status'] for frame in range(1, total_frames + 1)]
valid_count = statuses.count('ok')
if valid_count / total_frames < 0.98:
    print(f'Only {valid_count}/{total_frames} frames are valid before retry; run the missing-frame policy next.')
update_manifest(inference={'valid_frames': valid_count, 'total_frames': total_frames, 'status_counts': {value: statuses.count(value) for value in set(statuses)}})
print('Progress:', {value: statuses.count(value) for value in set(statuses)})

In [ ]:
progress = reconcile_progress(RUN_DIR, total_frames)
missing_frames = [frame for frame in range(1, total_frames + 1) if progress['frames'][str(frame)]['status'] != 'ok']
for frame in missing_frames:
    retry_dir = Path('/content/suhu_retry') / str(frame)
    if retry_dir.exists():
        shutil.rmtree(retry_dir)
    command = [env_python, 'inference.py', '--num_gpus', '1', '--pretrained_model', MODEL_NAME,
               '--img_path', RUN_DIR / 'frames', '--start', frame, '--end', '1', '--bbox_thr', '20',
               '--output_folder', retry_dir]
    run(command, cwd=SMPLERX_DIR / 'main', check=False)
    for folder, name in (('smplx', f'{frame:05d}_0.npz'), ('meta', f'{frame:05d}_0.json'), ('img', f'{frame:06d}.jpg')):
        source = retry_dir / folder / name
        destination_folder = 'overlays' if folder == 'img' else folder
        destination = RUN_DIR / destination_folder / name
        if source.exists():
            copy_once(source, destination)
progress = reconcile_progress(RUN_DIR, total_frames)
unresolved = [frame for frame in range(1, total_frames + 1) if progress['frames'][str(frame)]['status'] != 'ok']
update_manifest(missing_detection_retry={'attempted_frames': missing_frames, 'unresolved_frames': unresolved})
if unresolved:
    raise RuntimeError(f'Motion consolidation stopped. Unresolved frames after bbox_thr=20 retry: {unresolved}')
print('Every frame has valid SMPL-X parameters.')

In [ ]:
motion_paths = consolidate_motion(RUN_DIR, total_frames, config['processing_fps'])
with __import__('numpy').load(motion_paths['raw']) as archive:
    motion = {key: archive[key] for key in archive.files}
qa_report = analyze_motion(motion)
qa_report['frame_count'] = total_frames
qa_report['motion_files'] = {name: str(path) for name, path in motion_paths.items()}
qa_path = RUN_DIR / 'qa/suhu_motion_qa.json'
atomic_json(qa_path, qa_report)
if qa_report['invalid_value_count']:
    raise RuntimeError('Motion contains NaN or Inf values.')
flagged = {name: report['outlier_frames'] for name, report in qa_report['jumps'].items() if report['outlier_frames']}
print(json.dumps(qa_report, indent=2))
update_manifest(motion_qa=str(qa_path), motion_artifacts={name: str(path) for name, path in motion_paths.items()}, motion_outliers=flagged)
if flagged:
    confirmation = input(f'Compare these flagged frames with the source: {flagged}. Type CLEARED only if no jump changes the sign: ')
    if confirmation.strip() != 'CLEARED':
        raise RuntimeError('QA stopped: rerun the flagged frames before interpolation or smoothing.')
update_manifest(motion_visual_gate='cleared')

In [ ]:
overlay_video = RUN_DIR / 'suhu_smplerx_overlay.mp4'
source_vs_overlay = RUN_DIR / 'suhu_source_vs_smplx.mp4'
source_vs_overlay_slow = RUN_DIR / 'suhu_source_vs_smplx_half_speed.mp4'
run(['ffmpeg', '-y', '-framerate', config['processing_fps'], '-i', RUN_DIR / 'overlays/%06d.jpg',
     '-c:v', 'libx264', '-pix_fmt', 'yuv420p', overlay_video])
run(['ffmpeg', '-y', '-i', source_30fps, '-i', overlay_video, '-filter_complex',
     '[0:v][1:v]hstack=inputs=2[v]', '-map', '[v]', '-an', '-c:v', 'libx264', '-pix_fmt', 'yuv420p', source_vs_overlay])
run(['ffmpeg', '-y', '-i', source_vs_overlay, '-filter:v', 'setpts=2.0*PTS', '-an', '-c:v', 'libx264', '-pix_fmt', 'yuv420p', source_vs_overlay_slow])
update_manifest(comparison_artifacts={'overlay': str(overlay_video), 'source_vs_overlay': str(source_vs_overlay), 'source_vs_overlay_half_speed': str(source_vs_overlay_slow)})
from IPython.display import Video
display(Video(str(source_vs_overlay), embed=True))
print('Review normal and half speed: handshape, orientation, location, movement path, timing, and head/facial behaviour.')

In [ ]:
import textwrap
avatar_script = Path('/content/render_suhu_avatar.py')
avatar_script.write_text(textwrap.dedent(r'''
import json, os, sys
os.environ['PYOPENGL_PLATFORM'] = 'egl'
from pathlib import Path
import cv2
import numpy as np
import pyrender
import smplx
import torch
import trimesh

motion_path, model_dir, meta_dir, output_dir = map(Path, sys.argv[1:])
output_dir.mkdir(parents=True, exist_ok=True)
motion = dict(np.load(motion_path))
frame_count = len(motion['source_frame_number'])
metas = [json.loads((meta_dir / f'{frame:05d}_0.json').read_text()) for frame in range(1, frame_count + 1)]
focal = np.median(np.asarray([item['focal'] for item in metas]), axis=0)
princpt = np.median(np.asarray([item['princpt'] for item in metas]), axis=0)
model = smplx.create(str(model_dir), model_type='smplx', gender='neutral', ext='npz',
                     use_pca=False, num_betas=10, num_expression_coeffs=10)
material = pyrender.MetallicRoughnessMaterial(metallicFactor=0.0, roughnessFactor=0.75,
    alphaMode='OPAQUE', baseColorFactor=(0.18, 0.72, 0.78, 1.0))
renderer = pyrender.OffscreenRenderer(viewport_width=1280, viewport_height=720)
rotation = trimesh.transformations.rotation_matrix(np.radians(180), [1, 0, 0])
tensor = lambda name, index, flatten=False: torch.tensor(
    motion[name][index:index + 1].reshape(1, -1) if flatten else motion[name][index:index + 1], dtype=torch.float32)
for index in range(frame_count):
    with torch.no_grad():
        output = model(global_orient=tensor('global_orient', index), body_pose=tensor('body_pose', index, True),
            left_hand_pose=tensor('left_hand_pose', index, True), right_hand_pose=tensor('right_hand_pose', index, True),
            jaw_pose=tensor('jaw_pose', index), leye_pose=tensor('left_eye_pose', index),
            reye_pose=tensor('right_eye_pose', index), betas=tensor('betas', index),
            expression=tensor('expression', index), transl=tensor('translation', index))
    mesh = trimesh.Trimesh(output.vertices[0].numpy(), model.faces, process=False)
    mesh.apply_transform(rotation)
    scene = pyrender.Scene(bg_color=(0.025, 0.035, 0.08, 1.0), ambient_light=(0.45, 0.45, 0.45))
    scene.add(pyrender.Mesh.from_trimesh(mesh, material=material, smooth=True))
    scene.add(pyrender.IntrinsicsCamera(fx=float(focal[0]), fy=float(focal[1]),
        cx=float(princpt[0]), cy=float(princpt[1])))
    for position in ((0, -1, 1), (0, 1, 1), (1, 1, 2)):
        pose = np.eye(4); pose[:3, 3] = position
        scene.add(pyrender.DirectionalLight(color=np.ones(3), intensity=1.1), pose=pose)
    rgb, _ = renderer.render(scene)
    cv2.imwrite(str(output_dir / f'{index + 1:06d}.png'), cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR))
renderer.delete()
'''), encoding='utf-8')
avatar_frames = RUN_DIR / 'avatar/frames'
run([env_python, avatar_script, motion_paths['clean'], SMPLERX_DIR / 'common/utils/human_model_files/smplx', RUN_DIR / 'meta', avatar_frames])
avatar_video = RUN_DIR / 'avatar/suhu_avatar.mp4'
blind_video = DRIVE_ROOT / 'reviews/SUHU_blind_review.mp4'
three_way = RUN_DIR / 'avatar/suhu_three_way_comparison.mp4'
blind_video.parent.mkdir(parents=True, exist_ok=True)
run(['ffmpeg', '-y', '-framerate', config['processing_fps'], '-i', avatar_frames / '%06d.png',
     '-c:v', 'libx264', '-pix_fmt', 'yuv420p', avatar_video])
shutil.copy2(avatar_video, blind_video)
run(['ffmpeg', '-y', '-i', source_30fps, '-i', overlay_video, '-i', avatar_video, '-filter_complex',
     '[0:v][1:v][2:v]hstack=inputs=3[v]', '-map', '[v]', '-an', '-c:v', 'libx264', '-pix_fmt', 'yuv420p', three_way])
update_manifest(avatar_artifacts={'avatar': str(avatar_video), 'blind_review_copy': str(blind_video), 'three_way': str(three_way)})
display(Video(str(three_way), embed=True))
avatar_confirmation = input('After checking wrists, hand direction, fingers, mesh intersections, and framing, type AVATAR_CLEARED: ')
if avatar_confirmation.strip() != 'AVATAR_CLEARED':
    raise RuntimeError('Avatar gate stopped. Fix playback or rendering before blind BIM review.')
update_manifest(avatar_visual_gate='cleared')

In [ ]:
review_path = DRIVE_ROOT / 'reviews/SUHU_review.csv'
review_path.parent.mkdir(parents=True, exist_ok=True)
fields = ['reviewer_id', 'identified_word', 'handshape', 'orientation', 'location', 'movement', 'non_manual', 'comments']
if not review_path.exists():
    with review_path.open('w', newline='', encoding='utf-8') as stream:
        writer = csv.DictWriter(stream, fieldnames=fields)
        writer.writeheader()
        for reviewer in range(1, config['reviewer_count'] + 1):
            writer.writerow({'reviewer_id': f'R{reviewer}'})
with review_path.open(newline='', encoding='utf-8') as stream:
    reviews = list(csv.DictReader(stream))
if any(not row['identified_word'].strip() for row in reviews):
    print(f'Show only {blind_video.name} independently to three BIM-fluent reviewers, then fill {review_path} and rerun this cell.')
else:
    result = assess_reviews(reviews, config['sign_id'], config['reviewer_count'])
    result_path = RUN_DIR / 'qa/suhu_review_result.json'
    atomic_json(result_path, result)
    update_manifest(blind_review={'csv': str(review_path), 'result': str(result_path), **result})
    print(json.dumps(result, indent=2))
    if not result['passed']:
        raise RuntimeError('SUHU failed blind BIM validation; follow the failure routing in the README and do not scale.')
    print('SUHU PILOT PASSED. Proceed to the canary set, not the full vocabulary.')

## After SUHU passes

Run the unchanged full-body path on `Makan`, `Rumah`, `Doa`, `Jurukamera`, `Taman Negara Endau-Rompin`, `Ais`, `Wang tunai`, and `Balik ke rumah`. Route `1` and `A` to a separate hand-reconstruction experiment. Do not batch all 2,811 clips until the full-body canary passes BIM review and the hand-only route is defined.